# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefwerida28/Flyrank-ML-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Preparing the data**

In [23]:

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Connected successfully!")


Connected successfully!


In [24]:
df = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
    LIMIT 5
""").df()

df

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [25]:

staleness_df = con.sql("""
    SELECT
        f.report_date,
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        c.last_optimized_date,
        DATE_DIFF(
            'day',
            c.last_optimized_date,
            f.report_date
        ) AS days_since_last_update
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    ) AS f
    LEFT JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) AS c
    ON f.content_hash_id = c.content_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND c.last_optimized_date IS NOT NULL
""").df()

print("Rows:", len(staleness_df))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 1127183


**Signal 1 — Staleness: FALSE**

Reason: last_optimized_date is after the March report_date, so calculating staleness would use future information and would violate the no-future-information requirement.

In [26]:

import pandas as pd
staleness_df["staleness_bucket"] = pd.cut(
    staleness_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_table = (
    staleness_df
    .groupby("staleness_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

staleness_table

,staleness_bucket,n
0,0-30 days,0
1,31-90 days,0
2,91-180 days,0
3,181+ days,0


In [27]:
staleness_df["days_since_last_update"].describe()

,days_since_last_update
count,1.127183e+06
mean,-8.477753e+01
std,1.831376e+01
min,-1.270000e+02
25%,-9.900000e+01
50%,-8.500000e+01
75%,-7.100000e+01
max,-2.400000e+01


In [28]:
staleness_df["days_since_last_update"].isna().sum()

np.int64(0)

**Signal 2 — CTR vs position**
Verdict: CONFIRMED

Because better search positions generally have higher CTR, and we found some well-ranked pages with unusually low CTR.

In [29]:
ctr_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
""").df()

print("Rows:", len(ctr_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061


In [30]:
ctr_df["position_bucket"] = pd.cut(
    ctr_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"]
)


In [31]:
ctr_position_table = (
    ctr_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_ctr=("ctr", "mean")
    )
    .reset_index()
)

ctr_position_table

,position_bucket,n,avg_ctr
0,1-3,564173,0.004918
1,4-10,1456122,0.003473
2,11-20,519223,0.002770
3,21-50,631491,0.001638
4,51+,276863,0.000494


In [32]:
ctr_df = pd.merge(
    ctr_df,
    ctr_position_table[["position_bucket", "avg_ctr"]].rename(
        columns={"avg_ctr": "expected_ctr"}
    ),
    on="position_bucket",
    how="left",
)

ctr_df["ctr_ratio"] = ctr_df["ctr"] / ctr_df["expected_ctr"]

#Compare each page with its position's expected CTR
low_ctr_pages = (
    ctr_df[
        (ctr_df["gsc_avg_position"] <= 20) &
        (ctr_df["gsc_clicks"] > 0) &
        (ctr_df["ctr_ratio"] <= 0.5)
    ]
    .sort_values("ctr_ratio")
)

low_ctr_pages[
    [
        "content_hash_id",
        "gsc_avg_position",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "expected_ctr",
        "ctr_ratio"
    ]
].head(20)

,content_hash_id,gsc_avg_position,gsc_impressions,gsc_clicks,ctr,expected_ctr,ctr_ratio
3534021,content_44f34c0a90047651,0.083350,40084,1,0.000025,0.004918,0.005073
2857990,content_44f34c0a90047651,0.117814,30964,1,0.000032,0.004918,0.006567
43030,content_34a70fea29d15f24,2.764916,39003,2,0.000051,0.004918,0.010426
331411,content_1642f339bd6e7c8d,4.066814,24456,1,0.000041,0.003473,0.011775
3241302,content_44f34c0a90047651,0.142508,32756,2,0.000061,0.004918,0.012415
2822591,content_44f34c0a90047651,0.088955,30791,2,0.000065,0.004918,0.013207
3455447,content_44f34c0a90047651,0.238315,30573,2,0.000065,0.004918,0.013301
1704733,content_34a70fea29d15f24,3.626333,19223,1,0.000052,0.003473,0.014980
3174857,content_04fb6296acff6360,1.779776,11679,1,0.000086,0.004918,0.017410
1525495,content_1642f339bd6e7c8d,5.131579,16454,1,0.000061,0.003473,0.017501


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*o

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.